# Parte 2: Transformación a CNF y Resolución

En esta parte implementarás las transformaciones necesarias para convertir fórmulas a **Forma Normal Conjuntiva (CNF)**, y luego usarás el algoritmo de resolución (dado) para probar implicaciones.

## ¿Qué es CNF?

Una fórmula está en CNF si es una conjunción (∧) de cláusulas, donde cada cláusula es una disyunción (∨) de literales.

**Ejemplo:** `(p ∨ ¬q) ∧ (¬p ∨ r ∨ s) ∧ (q)`

## Pipeline de conversión a CNF

1. Eliminar bicondicionales (↔) → **TÚ**
2. Eliminar implicaciones (→) → **TÚ**
3. Mover negaciones hacia adentro (De Morgan) → **TÚ**
4. Eliminar dobles negaciones (¬¬) → **DADO**
5. Distribuir ∨ sobre ∧ → **TÚ**
6. Aplanar → **TÚ**

In [ ]:
import sys
sys.path.insert(0, '..')

from src.logic_core import Atom, Not, And, Or, Implies, Iff
from src.utils import formula_to_string

## Paso 1: Eliminar Bicondicionales

**Regla:** `A ↔ B` ≡ `(A → B) ∧ (B → A)`

Implementa `eliminate_iff` en `src/cnf_transform.py`.

In [ ]:
from src.cnf_transform import eliminate_iff

f = Iff(Atom('p'), Atom('q'))
result = eliminate_iff(f)
print(f"Original: {formula_to_string(f)}")
print(f"Sin ↔:    {formula_to_string(result)}")

## Paso 2: Eliminar Implicaciones

**Regla:** `A → B` ≡ `¬A ∨ B`

Implementa `eliminate_implication` en `src/cnf_transform.py`.

In [ ]:
from src.cnf_transform import eliminate_implication

f = Implies(Atom('p'), Atom('q'))
result = eliminate_implication(f)
print(f"Original: {formula_to_string(f)}")
print(f"Sin →:    {formula_to_string(result)}")

## Paso 3: Mover Negaciones hacia Adentro (De Morgan)

**Reglas:**
- `¬(A ∧ B)` ≡ `¬A ∨ ¬B`
- `¬(A ∨ B)` ≡ `¬A ∧ ¬B`

Implementa `push_negation_inward` en `src/cnf_transform.py`.

In [1]:
from src.cnf_transform import push_negation_inward

# De Morgan sobre And
f1 = Not(And(Atom('p'), Atom('q')))
r1 = push_negation_inward(f1)
print(f"¬(p ∧ q):  {formula_to_string(f1)}")
print(f"De Morgan: {formula_to_string(r1)}")

print()

# De Morgan sobre Or
f2 = Not(Or(Atom('p'), Atom('q')))
r2 = push_negation_inward(f2)
print(f"¬(p ∨ q):  {formula_to_string(f2)}")
print(f"De Morgan: {formula_to_string(r2)}")

ModuleNotFoundError: No module named 'src'

## Paso 4: Eliminar Dobles Negaciones (DADO)

**Regla:** `¬¬A` ≡ `A`

Esta función ya está implementada. Puedes estudiarla como referencia para las demás.

In [ ]:
from src.cnf_transform import eliminate_double_negation

f = Not(Not(Atom('p')))
result = eliminate_double_negation(f)
print(f"Original: {formula_to_string(f)}")
print(f"Sin ¬¬:   {formula_to_string(result)}")

## Paso 5: Distribuir ∨ sobre ∧

**Regla:** `A ∨ (B ∧ C)` ≡ `(A ∨ B) ∧ (A ∨ C)`

Esta es la transformación clave para obtener CNF: cada ∨ que contenga un ∧ adentro se "reparte".

Implementa `distribute_or_over_and` en `src/cnf_transform.py`.

In [ ]:
from src.cnf_transform import distribute_or_over_and

# p ∨ (q ∧ r) → (p ∨ q) ∧ (p ∨ r)
f = Or(Atom('p'), And(Atom('q'), Atom('r')))
result = distribute_or_over_and(f)
print(f"Original:    {formula_to_string(f)}")
print(f"Distribuido: {formula_to_string(result)}")

## Paso 6: Aplanar

**Reglas:**
- `(A ∧ B) ∧ C` ≡ `A ∧ B ∧ C`
- `(A ∨ B) ∨ C` ≡ `A ∨ B ∨ C`

El último paso: eliminar anidamientos innecesarios para obtener una CNF limpia.

Implementa `flatten` en `src/cnf_transform.py`.

In [ ]:
from src.cnf_transform import flatten

# And(And(a, b), c) → And(a, b, c)
f = And(And(Atom('a'), Atom('b')), Atom('c'))
result = flatten(f)
print(f"Original:  {formula_to_string(f)}")
print(f"Aplanado:  {formula_to_string(result)}")

## Pipeline completo: `to_cnf`

Una vez implementadas tus 5 funciones, el pipeline completo debería funcionar.

In [ ]:
from src.cnf_transform import to_cnf

# p → (q ∧ r) debería dar (¬p ∨ q) ∧ (¬p ∨ r)
f = Implies(Atom('p'), And(Atom('q'), Atom('r')))
cnf = to_cnf(f)
print(f"Original: {formula_to_string(f)}")
print(f"CNF:      {formula_to_string(cnf)}")

## Verificar con Resolución

Usa el módulo de resolución (dado) para probar que tus transformaciones son correctas.

In [ ]:
from src.resolution import resolution_prove

# Probar modus ponens: {p → q, p} ⊢ q
kb = [Implies(Atom('p'), Atom('q')), Atom('p')]
proved, steps = resolution_prove(kb, Atom('q'))
print(f"¿Probado? {proved}")
print("\nPasos:")
for step in steps:
    print(step)

In [ ]:
# Probar con el caso criminal
kb_criminal = [
    Implies(Not(Atom('llovía')), Not(Atom('jardín_mojado'))),
    Atom('jardín_mojado'),
]

proved, steps = resolution_prove(kb_criminal, Atom('llovía'))
print(f"¿Llovía? {proved}")
print("\nPasos:")
for step in steps:
    print(step)

In [ ]:
!cd .. && python -m pytest tests/test_cnf.py -v